# Capstone 1: Retail Sales Analysis & Insights

### Business Context

You are a data analyst at "UrbanCart," a multi-category retail company. Management wants a clear picture of last year's performance and actionable recommendations — the single most common real-world deliverable this entire course has been preparing you to produce.

### Step 1: Generate/Load and Explore the Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

np.random.seed(42)
sns.set_style("whitegrid")

n = 5000
start_date = datetime(2024, 1, 1)
dates = [start_date + timedelta(days=np.random.randint(0, 365)) for _ in range(n)]

df = pd.DataFrame({
    "order_id": range(10001, 10001 + n),
    "order_date": dates,
    "region": np.random.choice(["North", "South", "East", "West"], n, p=[0.3, 0.25, 0.25, 0.2]),
    "category": np.random.choice(["Electronics", "Apparel", "Home", "Grocery", "Beauty"], n),
    "quantity": np.random.randint(1, 6, n),
    "unit_price": np.round(np.random.uniform(100, 5000, n), 0),
    "customer_id": np.random.randint(1000, 2000, n),
    "channel": np.random.choice(["Online", "Store", "App"], n, p=[0.5, 0.3, 0.2])
})
df["revenue"] = df["quantity"] * df["unit_price"]
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.to_period("M")

print(df.head())
print("\nShape:", df.shape)
print("\nDate range:", df["order_date"].min().date(), "->", df["order_date"].max().date())

Notice this step alone exercises Volume 02 (DataFrame construction, `pd.to_datetime`, `.dt` accessor) and Volume 01's `range()`/list comprehension habits — every real project starts here, with construction or loading followed immediately by a shape/date-range sanity check, exactly as Volume 02's Chapter 2 taught as an automatic first move.

### Step 2: Headline KPIs

In [ ]:
total_revenue = df["revenue"].sum()
total_orders = df["order_id"].nunique()
aov = total_revenue / total_orders                    # Average Order Value — a standard retail KPI
unique_customers = df["customer_id"].nunique()

print("="*40)
print("URBANCART -- ANNUAL SNAPSHOT")
print("="*40)
print(f"Total Revenue      : ₹{total_revenue:,.0f}")
print(f"Total Orders       : {total_orders:,}")
print(f"Average Order Value: ₹{aov:,.0f}")
print(f"Unique Customers   : {unique_customers:,}")

**AOV (Average Order Value)** is worth knowing as a standard retail/e-commerce metric even beyond this exercise — it's frequently a more actionable lever than total revenue alone, since a business can grow revenue either by acquiring more customers/orders *or* by increasing what each order is worth (via bundling, upselling, minimum-spend free-shipping thresholds), and AOV is precisely the number that tracks the second lever.

### Step 3: Trend and Composition Analysis (Volume 03 in Action)

In [ ]:
monthly = df.groupby("month")["revenue"].sum()
monthly.index = monthly.index.astype(str)

plt.figure(figsize=(12, 5))
monthly.plot(kind="line", marker="o")
plt.title("Monthly Revenue Trend")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

region_rev = df.groupby("region")["revenue"].sum().sort_values(ascending=False)
cat_rev = df.groupby("category")["revenue"].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
region_rev.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Revenue by Region")
cat_rev.plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Revenue by Category")
plt.tight_layout()
plt.show()

print("Top Region:", region_rev.index[0], f"(₹{region_rev.iloc[0]:,.0f})")
print("Top Category:", cat_rev.index[0], f"(₹{cat_rev.iloc[0]:,.0f})")

### Step 4: RFM Customer Segmentation — A Genuinely Important Real-World Technique

In [ ]:
snapshot = df["order_date"].max() + timedelta(days=1)
rfm = df.groupby("customer_id").agg(
    recency=("order_date", lambda x: (snapshot - x.max()).days),     # days since last purchase
    frequency=("order_id", "count"),                                   # number of orders
    monetary=("revenue", "sum")                                        # total spend
).reset_index()

rfm["R_score"] = pd.qcut(rfm["recency"], 4, labels=[4,3,2,1])          # LOWER recency (more recent) -> HIGHER score
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1,2,3,4])
rfm["M_score"] = pd.qcut(rfm["monetary"], 4, labels=[1,2,3,4])
rfm["RFM_Score"] = rfm["R_score"].astype(int) + rfm["F_score"].astype(int) + rfm["M_score"].astype(int)

def segment(score):
    if score >= 10: return "Champions"
    elif score >= 8: return "Loyal"
    elif score >= 6: return "Potential"
    else: return "At Risk"

rfm["Segment"] = rfm["RFM_Score"].apply(segment)
print(rfm["Segment"].value_counts())
print("\nAverage monetary by segment:\n", rfm.groupby("Segment")["monetary"].mean().round(0))

**RFM (Recency, Frequency, Monetary) segmentation** is one of the most widely used customer analytics techniques in real retail and marketing teams, precisely because it requires no machine learning at all — just three simple, business-intuitive metrics computed directly from transaction history, then bucketed with `pd.qcut` into quartile scores. Compare this to Volume 05's K-Means clustering: both produce customer segments, but RFM's segments are immediately interpretable to any business stakeholder ("Champions bought recently, often, and spend a lot") without needing to explain an algorithm at all — a genuinely useful technique to have as a fast, defensible alternative to unsupervised ML when speed and stakeholder buy-in matter more than algorithmic sophistication.

`pd.qcut` (quantile cut) — worth remembering as distinct from `pd.cut` (which bins by fixed value ranges) — splits a numeric column into groups with *equal counts* of observations, rather than equal-width ranges; this is exactly the right tool when you want "top 25%, next 25%," etc., regardless of the underlying value distribution.

### Step 5: The Executive Summary — The Actual Deliverable

```
UrbanCart FY Performance — Executive Summary

1. OVERALL: Total revenue was ₹X across Y orders, with an average order value of ₹Z.
2. GEOGRAPHIC: [Top region] contributed the highest share of revenue; [weakest region]
   is underperforming and warrants investigation into local demand or execution issues.
3. CATEGORY: [Top category] remains the primary revenue engine; [a smaller category]
   shows growth potential worth testing with increased marketing investment.
4. CUSTOMERS: "Champions" (top RFM segment) generate disproportionate revenue relative
   to their numbers — protecting this segment with a loyalty program is higher-leverage
   than acquiring new customers of unknown value.
5. RECOMMENDATIONS:
   - Reallocate marketing budget toward the highest-ROI region/category combination.
   - Launch a win-back campaign specifically targeting the "At Risk" segment before they
     churn entirely.
   - Test higher-AOV tactics (bundling, free-shipping thresholds) specifically in the
     App channel, which typically has the most room to grow order value.
```

💡 **This five-point structure — Overall, Geographic, Category, Customer, Recommendations — is a genuinely reusable template** for almost any business performance review you'll ever write, regardless of the specific industry or metrics involved. Notice it leads with the numbers but ends with *specific, assignable actions* — a report that stops at "here's what happened" without translating it into "here's what to do about it" has left the most valuable part of the analysis undone.

---